In [ ]:
# Mount Google Drive
from google.colab import drive
import os

# Disable Weights & Biases tracking
os.environ["WANDB_DISABLED"] = "true"

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Define directories and file paths
main_dir = "/content/drive/MyDrive/ITCS6156_BERT"
result_dir = os.path.join(main_dir, "Results")
data_dir = os.path.join(main_dir, "Data_Multilabel")
data_path = os.path.join(data_dir, "movies.csv")

In [ ]:
import pandas as pd

# Load data
df_movies = pd.read_csv(data_path)

# Pre-process data
df_movies = df_movies[['MOVIES', 'GENRE', 'ONE-LINE']]
df_movies = df_movies.applymap(lambda x: x.lower() if isinstance(x, str) else x)
df_movies = df_movies.drop_duplicates(subset=["MOVIES", "GENRE", "ONE-LINE"])
df_movies = df_movies.dropna(subset=['GENRE'])
df_movies = df_movies.apply(lambda col: col.str.replace('\n', '', regex=True) if col.dtype == 'object' else col)
df_movies = df_movies[df_movies['ONE-LINE'] != "add a plot"]
df_movies = df_movies.reset_index(drop=True)

# Combine title and description for text input
df_movies["TEXT"] = df_movies["MOVIES"] + " " + df_movies["ONE-LINE"]

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

df_movies['GENRE_LIST'] = df_movies['GENRE'].apply(lambda x: [genre.strip() for genre in x.split(",")])

mlb = MultiLabelBinarizer()
encoded_genres = mlb.fit_transform(df_movies['GENRE_LIST'])

genre_encoded_df = pd.DataFrame(encoded_genres, columns=mlb.classes_)
df_movies = pd.concat([df_movies, genre_encoded_df], axis=1)

In [ ]:
from sklearn.model_selection import train_test_split

# Split training data into train and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(df_movies["TEXT"], df_movies["GENRE_LIST"], test_size=0.2, random_state=42)

In [ ]:
print(f"Training set size: {train_texts.shape[0]}")
print(f"Test set size: {val_texts.shape[0]}")

In [ ]:
from transformers import DistilBertTokenizer

# Initialize tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize text data
movies_encodings = tokenizer(list(df_movies["TEXT"]), truncation=True, padding=True, max_length=256)

In [ ]:
import tensorflow as tf

# Tokenize train and validation data
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=256)

# Create TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), tf.convert_to_tensor(mlb.transform(train_labels), dtype=tf.float32))).batch(16)
val_dataset = tf.data.Dataset.from_tensor_slices((dict(val_encodings), tf.convert_to_tensor(mlb.transform(val_labels), dtype=tf.float32))).batch(16)

train_dataset = train_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
from transformers import TFDistilBertForSequenceClassification, DistilBertConfig

# Load pre-trained DistilBERT model for multi-label classification
num_labels = len(mlb.classes_)
config = DistilBertConfig.from_pretrained("distilbert-base-uncased", num_labels=num_labels, hidden_dropout_prob=0.2, problem_type="multi_label_classification")

model = TFDistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", config=config)

In [ ]:
# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=7e-5)
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [ ]:
# Train the model
history = model.fit(train_dataset, validation_data=val_dataset, epochs=3)

In [ ]:
# Predict on validation data
predictions = model.predict(val_dataset)

# Convert predictions back to genre labels
predicted_labels = mlb.inverse_transform((predictions.logits > 0.5).astype(int))

# print(predicted_labels)

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix

# Convert predictions to binary format
binary_predictions = (predictions.logits > 0.5).astype(int)

# Convert validation labels to binary format
binary_val_labels = mlb.transform(val_labels)

# Compute confusion matrices for each label
confusion_matrices = multilabel_confusion_matrix(binary_val_labels, binary_predictions)

In [ ]:
class_labels = mlb.classes_
for label, matrix in zip(class_labels, confusion_matrices):
    print(f"< {label.capitalize()} >")
    print(pd.DataFrame(matrix,
                       index=["Actual Negative", "Actual Positive"],
                       columns=["Predicted Negative", "Predicted Positive"]))
    print()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Confusion Matrices")

# Plot confusion matrix for a specific label
for idx, genre in enumerate(mlb.classes_):
    cm = confusion_matrices[idx]
    plt.figure(figsize=(2, 1))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["N", "P"], yticklabels=["N", "P"], annot_kws={"size": 8})
    plt.title(genre.capitalize(), fontsize=10)
    plt.xlabel("Predicted", fontsize=10)
    plt.ylabel("True", fontsize=10)
    plt.show()

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
accuracy = accuracy_score(binary_val_labels, binary_predictions)

print("\nClassification Report:")
print(classification_report(binary_val_labels, binary_predictions, target_names=mlb.classes_))
print(f"\nAccuracy: {accuracy:.2f}\n\n")